In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.4 Continuous batching

A GPU is fastest when fed many sequences at once. **Continuous batching** packs
multiple requests into each forward pass and lets requests join or leave mid-flight,
instead of waiting for the slowest in a static batch. The core move is one forward
over a padded, masked batch.

In [ ]:
from pocketlm import train_tiny, pick_config, pad_batch

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
prompts = ["The ", "ROMEO", "a"]
ids = [tok.encode(p) for p in prompts]
padded, mask = pad_batch(ids, pad_id=0)
logits, _ = model(padded)
last = mask.sum(1) - 1                                  # each prompt's final real token
nxt = logits[torch.arange(len(prompts)), last].argmax(-1)
for p, i in zip(prompts, nxt):
    print(f"{p!r:>8} -> next char {tok.itos[int(i)]!r}")

One forward served three different-length prompts. A real server keeps a running
batch, admitting new requests and retiring finished ones every step, to keep
utilization high and latency low.

In [ ]:
assert tuple(nxt.shape) == (len(prompts),)